## CELL 1 — Config & Seed

In [1]:
"""
NOTEBOOK 1B.9 + 1B.10 - Ensemble 5 seeds + TTA horizontal flip
Fase 1B Ronde 4 - IndoMeme-XAI

Best config dari 1B.3: ICAM + SupCon (lambda=0.3) + curriculum.
Run dengan 5 seeds, ensemble probabilitas, evaluasi dengan dan tanpa TTA.
Target: Macro-F1 >= 0.83 (single ensemble), >= 0.85 (ensemble + TTA).
"""

from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input/datasets/reynaldaryansyah")
ROOT_DIR     = KAGGLE_INPUT / "xai-dataset" / "data"
CSV_PATH     = ROOT_DIR / "processed" / "all_splits.csv"
IMG_DIR      = ROOT_DIR / "images"

OUTPUT_DIR = Path("/kaggle/working/exp_1B9_ensemble")
PROBS_DIR  = OUTPUT_DIR / "probs"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
for d in [OUTPUT_DIR, PROBS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = dict(
    exp_id           = "1B.9_ensemble_5seeds",

    # Ensemble seeds
    seeds            = [42, 123, 2024, 7, 999],

    text_model_name  = "indolem/indobertweet-base-uncased",
    image_model_name = "google/vit-base-patch16-224-in21k",
    freeze_n_layers  = 4,

    topic_emb_dim    = 32,
    head_hidden      = 256,
    dropout          = 0.3,

    icam_num_heads   = 8,
    icam_dropout     = 0.1,

    # SupCon
    supcon_lambda          = 0.3,
    supcon_temperature     = 0.07,
    supcon_proj_hidden     = 128,
    supcon_proj_out        = 128,
    supcon_warmup_epochs   = 3,

    max_seq_len      = 128,

    # Training (lebih efisien dari 1B.3 berdasarkan observasi best_epoch=8)
    batch_size       = 32,
    grad_accum       = 2,
    num_epochs       = 15,             # turun dari 20
    early_stop_pat   = 6,               # turun dari 7
    grad_clip        = 1.0,

    lr_encoder       = 1e-5,
    lr_fusion        = 3e-5,
    lr_head          = 5e-5,
    weight_decay     = 0.01,
    warmup_ratio     = 0.10,

    focal_gamma      = 2.0,
    label_smoothing  = 0.1,
    class_weights    = [0.76, 1.46],

    # Threshold search range diperketat (1B.3 optimal 0.57)
    thr_search_lo    = 0.40,
    thr_search_hi    = 0.65,
    thr_search_step  = 0.01,
)

print(f"Experiment: {CONFIG['exp_id']}")
print(f"Seeds     : {CONFIG['seeds']}")
print(f"Output dir: {OUTPUT_DIR}")

Experiment: 1B.9_ensemble_5seeds
Seeds     : [42, 123, 2024, 7, 999]
Output dir: /kaggle/working/exp_1B9_ensemble


## CELL 2 — Imports

In [2]:
import os, json, random, time, math, re, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from PIL import Image
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, accuracy_score,
    roc_auc_score, classification_report, confusion_matrix,
)
from torchvision import transforms
from transformers import (
    AutoTokenizer, AutoModel, ViTImageProcessor, ViTModel,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda  |  PyTorch: 2.10.0+cu128
GPU: Tesla T4


## CELL 3 — Slang Dictionary

In [3]:
SLANG_DICT = {
    "gw":"saya","gue":"saya","gua":"saya","guwe":"saya",
    "lu":"kamu","lo":"kamu","loe":"kamu","elo":"kamu","elu":"kamu",
    "ane":"saya","ente":"kamu","agan":"kamu","sis":"kakak",
    "bro":"kawan","bray":"kawan","bre":"kawan","cuy":"kawan",
    "klian":"kalian","kmu":"kamu","kamuh":"kamu",
    "aq":"aku","akoh":"aku","akuh":"aku",
    "gak":"tidak","ga":"tidak","gk":"tidak","nggak":"tidak",
    "ngga":"tidak","ngk":"tidak","kgk":"tidak","tdk":"tidak",
    "engga":"tidak","enggak":"tidak","kga":"tidak",
    "yg":"yang","dgn":"dengan","dg":"dengan","krn":"karena",
    "krna":"karena","utk":"untuk","tp":"tapi","tpi":"tapi",
    "klo":"kalau","klau":"kalau","kl":"kalau","klu":"kalau",
    "spy":"supaya","ato":"atau",
    "bgt":"banget","bgtt":"banget","bngt":"banget",
    "bnr":"benar","bener":"benar",
    "emg":"memang","emang":"memang","mmg":"memang",
    "skrg":"sekarang","skrng":"sekarang",
    "td":"tadi","tdi":"tadi","bsk":"besok",
    "knp":"kenapa","gmn":"bagaimana","gimana":"bagaimana",
    "gmana":"bagaimana","kyk":"seperti","kayak":"seperti","kek":"seperti",
    "udh":"sudah","udah":"sudah","uda":"sudah","dah":"sudah",
    "blm":"belum","blom":"belum",
    "lg":"lagi","lgi":"lagi","msh":"masih","msi":"masih",
    "jd":"jadi","jdi":"jadi",
    "tau":"tahu","taw":"tahu","tw":"tahu",
    "liat":"lihat","ngeliat":"melihat","denger":"dengar",
    "ngomong":"bicara","blng":"bilang","bilg":"bilang",
    "aja":"saja","doang":"saja","doank":"saja",
    "sih":"","deh":"","dong":"","kok":"","lah":"",
    "loh":"","lho":"","yah":"",
    "iy":"iya","iyaa":"iya","iye":"iya","yup":"iya","yoi":"iya",
    "org":"orang","orng":"orang","orgnya":"orangnya",
    "dr":"dari","dri":"dari","kpd":"kepada",
    "thd":"terhadap","ttg":"tentang",
    "abis":"habis","trs":"terus","trus":"terus",
    "tmn":"teman","temen":"teman","tman":"teman",
    "rmh":"rumah","kntr":"kantor","krj":"kerja",
    "ank":"anak","indo":"indonesia",
    "cwo":"cowok","cwe":"cewek","cwok":"cowok","cwek":"cewek",
    "wkwk":"haha","wkwkwk":"haha","wkwkwkwk":"haha","wkwkwkwkwk":"haha",
    "hahaha":"haha","hahahaha":"haha",
    "hehehe":"hehe","hehehehe":"hehe",
    "anjir":"anjing","anjay":"anjing","anjrit":"anjing",
    "anjr":"anjing","anying":"anjing","njir":"anjing",
    "bgsd":"bangsat","bgst":"bangsat",
    "bjir":"anjing","bjg":"bajingan",
    "kntl":"kontol","mmk":"memek","tai":"tahi",
}
print(f"Slang dict entries: {len(SLANG_DICT)}")

Slang dict entries: 149


## CELL 4 — clean_ocr

In [4]:
URL_PATTERN     = re.compile(r"(https?://\S+|www\.\S+|\S+\.(?:com|co\.id|net|org|id|tv|me)\b\S*)", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
ALLOWED_PUNCT   = re.compile(r"[^\w\s\.,!?\-'\"]+", re.UNICODE)
MULTI_SPACE     = re.compile(r"\s+")

def _normalize_unicode(t): return unicodedata.normalize("NFKC", t)

def _normalize_slang(text, slang_dict=SLANG_DICT):
    out = []
    for tok in text.split():
        prefix, suffix, core = "", "", tok
        while core and not core[0].isalnum(): prefix += core[0]; core = core[1:]
        while core and not core[-1].isalnum(): suffix = core[-1] + suffix; core = core[:-1]
        out.append(f"{prefix}{slang_dict.get(core.lower(), core)}{suffix}")
    return " ".join(out)

def clean_ocr(text):
    if text is None: return ""
    if isinstance(text, float) and pd.isna(text): return ""
    text = _normalize_unicode(str(text)).lower()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = ALLOWED_PUNCT.sub(" ", text)
    text = _normalize_slang(text)
    return MULTI_SPACE.sub(" ", text).strip()

## CELL 5 — format_input

In [5]:
SEP_TOKEN = "[SEP]"

def _safe_str(x):
    if x is None: return ""
    if isinstance(x, float) and pd.isna(x): return ""
    return str(x).strip()

def format_input(topic, caption, ocr_cleaned, sep=SEP_TOKEN):
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        topic_str = "unknown"
    elif isinstance(topic, (list, tuple, set)):
        items = sorted({str(t).lower().strip() for t in topic if t})
        topic_str = ",".join(items) if items else "unknown"
    else:
        topic_str = str(topic).lower().strip() or "unknown"
    return f"[{topic_str}] {_safe_str(caption)} {sep} {_safe_str(ocr_cleaned)}".strip()

## CELL 6 — Load Data

In [6]:
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}")
print(df["split"].value_counts().to_dict())

LABEL_MAP = {"not hate": 0, "hate": 1}
df["label"] = df["hate_final"].map(LABEL_MAP)
assert df["label"].notna().all()

df["ocr_cleaned"] = df["ocr"].apply(clean_ocr)
df["text_input"]  = df.apply(lambda r: format_input(r["topic"], r["caption"], r["ocr_cleaned"]), axis=1)

def resolve_image_path(row):
    raw = row.get("local_image_path") or row.get("image_path")
    if raw is None or (isinstance(raw, float) and pd.isna(raw)): return None
    fname = Path(str(raw).replace("\\","/")).name
    return str(IMG_DIR / row["split"] / fname)

df["full_image_path"] = df.apply(resolve_image_path, axis=1)

Total samples: 5023
{'train': 3415, 'test': 1005, 'validation': 603}


## CELL 7 — Topic Multi-Hot 

In [7]:
TOPIC_LIST = [
    "political","national origin","individual","institutions",
    "gender","religion","social sub-groups","unknown",
]
TOPIC2IDX = {t: i for i, t in enumerate(TOPIC_LIST)}
N_TOPICS  = len(TOPIC_LIST)

def topic_to_multihot(topic):
    vec = np.zeros(N_TOPICS, dtype=np.float32)
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        vec[TOPIC2IDX["unknown"]] = 1.0; return vec
    parts = [p.strip().lower() for p in str(topic).split(",") if p.strip()]
    found = False
    for p in parts:
        if p in TOPIC2IDX:
            vec[TOPIC2IDX[p]] = 1.0; found = True
    if not found: vec[TOPIC2IDX["unknown"]] = 1.0
    return vec

df["topic_multihot"] = df["topic"].apply(topic_to_multihot)

## CELL 8 — Tokenizer & Image Processor

In [8]:
tokenizer       = AutoTokenizer.from_pretrained(CONFIG["text_model_name"])
image_processor = ViTImageProcessor.from_pretrained(CONFIG["image_model_name"])

IMG_MEAN = image_processor.image_mean
IMG_STD  = image_processor.image_std
IMG_SIZE = image_processor.size["height"] if isinstance(image_processor.size, dict) else 224
print(f"Image: size={IMG_SIZE}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Image: size=224


## CELL 9 — Image Transforms 

In [9]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

## CELL 10 — Dataset Class

In [10]:
class IndoMemeDataset(Dataset):
    def __init__(self, df_split, tokenizer, transform, max_seq_len=128):
        self.df          = df_split.reset_index(drop=True)
        self.tokenizer   = tokenizer
        self.transform   = transform
        self.max_seq_len = max_seq_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["full_image_path"]).convert("RGB")
        img_tensor = self.transform(img)
        enc = self.tokenizer(row["text_input"], padding="max_length", truncation=True,
                             max_length=self.max_seq_len, return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values":   img_tensor,
            "topic":          torch.from_numpy(row["topic_multihot"]),
            "label":          torch.tensor(int(row["label"]), dtype=torch.long),
            "idx":            int(row["idx"]),
        }

## CELL 11 — Build Datasets (train + train_eval + val + test)

In [11]:
df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "validation"].copy() if "validation" in df["split"].unique() else df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()

ds_train      = IndoMemeDataset(df_train, tokenizer, TRAIN_TRANSFORM, CONFIG["max_seq_len"])
ds_train_eval = IndoMemeDataset(df_train, tokenizer, EVAL_TRANSFORM,  CONFIG["max_seq_len"])
ds_val        = IndoMemeDataset(df_val,   tokenizer, EVAL_TRANSFORM,  CONFIG["max_seq_len"])
ds_test       = IndoMemeDataset(df_test,  tokenizer, EVAL_TRANSFORM,  CONFIG["max_seq_len"])

print(f"Train: {len(ds_train)}  Val: {len(ds_val)}  Test: {len(ds_test)}")

# Pre-compute sample weights utk WeightedRandomSampler (dipakai di tiap seed)
labels_train = df_train["label"].values
class_count  = np.bincount(labels_train, minlength=2)
sample_w     = 1.0 / class_count[labels_train]

# Val/test/train_eval loader: deterministik, dibuat 1x (tidak dipengaruhi seed)
dl_val        = DataLoader(ds_val,        batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
dl_test       = DataLoader(ds_test,       batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
dl_train_eval = DataLoader(ds_train_eval, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

print(f"Class count train: {class_count.tolist()}")

Train: 3415  Val: 603  Test: 1005
Class count train: [2242, 1173]


## CELL 12 — ICAM Module

In [12]:
class ICAMFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.t2i_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.gate_relevant    = nn.Linear(2 * hidden_dim, hidden_dim)
        self.gate_incongruent = nn.Linear(2 * hidden_dim, hidden_dim)
        self.fusion_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm  = nn.LayerNorm(hidden_dim)
        self.save_attn = False

    def forward(self, text_h, image_h, text_mask=None):
        text_kpm = (~text_mask.bool()) if text_mask is not None else None
        t2i_out, _ = self.t2i_attn(query=text_h, key=image_h, value=image_h,
                                    key_padding_mask=None,
                                    need_weights=self.save_attn, average_attn_weights=True)
        i2t_out, _ = self.i2t_attn(query=image_h, key=text_h, value=text_h,
                                    key_padding_mask=text_kpm,
                                    need_weights=self.save_attn, average_attn_weights=True)
        h_image_pooled = i2t_out.mean(dim=1)

        text_norm  = F.normalize(text_h,  dim=-1)
        image_norm = F.normalize(image_h, dim=-1)
        sim         = torch.bmm(text_norm, image_norm.transpose(1, 2))
        incongruity = 1.0 - sim.abs()
        inc_w       = F.softmax(incongruity, dim=-1)
        h_inc       = torch.bmm(inc_w, image_h)

        if text_mask is not None:
            m = text_mask.unsqueeze(-1).float()
            h_inc_pooled = (h_inc * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
        else:
            h_inc_pooled = h_inc.mean(dim=1)

        h_text = text_h[:, 0]
        g1 = torch.sigmoid(self.gate_relevant(   torch.cat([h_text, h_image_pooled], dim=-1)))
        g2 = torch.sigmoid(self.gate_incongruent(torch.cat([h_text, h_inc_pooled  ], dim=-1)))

        fused   = h_text + g1 * h_image_pooled + g2 * h_inc_pooled
        h_fused = self.layer_norm(self.fusion_proj(fused))
        return h_fused, g1.mean().item(), g2.mean().item()

## CELL 13 — Full Model 

In [13]:
class ICAMHateModelSupCon(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.text_encoder  = AutoModel.from_pretrained(cfg["text_model_name"])
        self.image_encoder = ViTModel.from_pretrained(cfg["image_model_name"])
        self._freeze_bottom(self.text_encoder.encoder.layer,  cfg["freeze_n_layers"])
        self._freeze_bottom(self.image_encoder.encoder.layer, cfg["freeze_n_layers"])
        for p in self.text_encoder.embeddings.parameters():  p.requires_grad = False
        for p in self.image_encoder.embeddings.parameters(): p.requires_grad = False

        hidden = self.text_encoder.config.hidden_size
        self.icam = ICAMFusion(hidden_dim=hidden, num_heads=cfg["icam_num_heads"], dropout=cfg["icam_dropout"])
        self.supcon_proj = nn.Sequential(
            nn.Linear(hidden, cfg["supcon_proj_hidden"]),
            nn.ReLU(),
            nn.Linear(cfg["supcon_proj_hidden"], cfg["supcon_proj_out"]),
        )
        self.topic_proj = nn.Sequential(nn.Linear(N_TOPICS, cfg["topic_emb_dim"]), nn.GELU())
        head_in = hidden + cfg["topic_emb_dim"]
        self.head = nn.Sequential(
            nn.Linear(head_in, cfg["head_hidden"]), nn.GELU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["head_hidden"], 2),
        )

    @staticmethod
    def _freeze_bottom(layers, n):
        for i, layer in enumerate(layers):
            if i < n:
                for p in layer.parameters(): p.requires_grad = False

    def forward(self, input_ids, attention_mask, pixel_values, topic):
        t_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        i_out = self.image_encoder(pixel_values=pixel_values)
        h_fused, g1m, g2m = self.icam(t_out.last_hidden_state, i_out.last_hidden_state, attention_mask)
        supcon_features = F.normalize(self.supcon_proj(h_fused), dim=-1)
        topic_emb = self.topic_proj(topic)
        logits    = self.head(torch.cat([h_fused, topic_emb], dim=-1))
        return logits, supcon_features, g1m, g2m

## CELL 14 — Focal Loss

In [14]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, num_classes=2):
        super().__init__()
        self.gamma, self.label_smoothing, self.num_classes = gamma, label_smoothing, num_classes
        if alpha is not None:
            self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=-1); probs = log_probs.exp()
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.label_smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.label_smoothing)
        pt      = probs.gather(1, target.unsqueeze(1)).squeeze(1).clamp(min=1e-8)
        focal_w = (1.0 - pt) ** self.gamma
        ce      = -(true_dist * log_probs).sum(dim=-1)
        loss    = focal_w * ce
        if self.alpha is not None:
            loss = self.alpha.to(logits.device)[target] * loss
        return loss.mean()

focal_criterion = FocalLoss(
    gamma=CONFIG["focal_gamma"], alpha=CONFIG["class_weights"],
    label_smoothing=CONFIG["label_smoothing"],
)

## CELL 15 — SupCon Loss

In [15]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device; B = features.size(0)
        sim = torch.matmul(features, features.T) / self.temperature
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - sim_max.detach()
        labels = labels.contiguous().view(-1, 1)
        mask_pos = (labels == labels.T).float().to(device)
        eye      = torch.eye(B, device=device)
        mask_pos = mask_pos - eye
        mask_all = 1.0 - eye
        exp_sim   = torch.exp(sim) * mask_all
        log_prob  = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)
        n_pos = mask_pos.sum(dim=1)
        mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1) / n_pos.clamp(min=1.0)
        valid = (n_pos > 0).float()
        loss  = -(mean_log_prob_pos * valid).sum() / valid.sum().clamp(min=1.0)
        return loss

supcon_criterion = SupConLoss(temperature=CONFIG["supcon_temperature"])

## CELL 16 — Optimizer Builder 

In [16]:
def build_optimizer_scheduler(model, cfg, num_training_steps):
    enc_p, fusion_p, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith("text_encoder") or n.startswith("image_encoder"): enc_p.append(p)
        elif n.startswith("icam"): fusion_p.append(p)
        else: head_p.append(p)
    groups = [
        {"params": enc_p,    "lr": cfg["lr_encoder"], "weight_decay": cfg["weight_decay"]},
        {"params": fusion_p, "lr": cfg["lr_fusion"],  "weight_decay": cfg["weight_decay"]},
        {"params": head_p,   "lr": cfg["lr_head"],    "weight_decay": cfg["weight_decay"]},
    ]
    optimizer = torch.optim.AdamW(groups)
    num_warmup = int(cfg["warmup_ratio"] * num_training_steps)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=num_warmup, num_training_steps=num_training_steps)
    return optimizer, scheduler

## CELL 17 — Metrics & Threshold Search 

In [17]:
def compute_metrics(y_true, y_pred, y_prob_hate=None):
    out = {}
    out["macro_f1"]    = f1_score(y_true, y_pred, average="macro")
    out["weighted_f1"] = f1_score(y_true, y_pred, average="weighted")
    out["accuracy"]    = accuracy_score(y_true, y_pred)
    p, r, f, _         = precision_recall_fscore_support(y_true, y_pred, labels=[0,1], zero_division=0)
    out["nothate_p"], out["nothate_r"], out["nothate_f1"] = p[0], r[0], f[0]
    out["hate_p"],    out["hate_r"],    out["hate_f1"]    = p[1], r[1], f[1]
    if y_prob_hate is not None and len(set(y_true)) > 1:
        out["roc_auc"] = roc_auc_score(y_true, y_prob_hate)
    return out

def search_threshold(y_true, y_prob_hate, lo, hi, step):
    best = (-1, 0.5)
    for thr in tqdm(np.arange(lo, hi + 1e-9, step), desc="threshold search", leave=False):
        f = f1_score(y_true, (y_prob_hate >= thr).astype(int), average="macro")
        if f > best[0]:
            best = (f, float(thr))
    return best

## CELL 18 — Train / Eval Functions + evaluate_get_probs (TTA-aware)

In [18]:
@torch.no_grad()
def evaluate(model, loader, threshold=0.5, desc="eval"):
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        pixel_values   = batch["pixel_values"].to(DEVICE, non_blocking=True)
        topic          = batch["topic"].to(DEVICE, non_blocking=True)
        with autocast(device_type="cuda"):
            logits, _, _, _ = model(input_ids, attention_mask, pixel_values, topic)
        all_logits.append(logits.float().cpu()); all_labels.append(batch["label"])
    logits = torch.cat(all_logits); labels = torch.cat(all_labels).numpy()
    probs  = F.softmax(logits, dim=-1).numpy()
    prob_hate = probs[:, 1]
    preds  = (prob_hate >= threshold).astype(int)
    metrics = compute_metrics(labels, preds, prob_hate); metrics["threshold"] = threshold
    return metrics, prob_hate, labels


@torch.no_grad()
def evaluate_get_probs(model, loader, tta=False, desc="probs"):
    """
    Mengembalikan (probs_no_flip, probs_flip, labels) jika tta=True,
    atau (probs_no_flip, labels) jika tta=False.
    Order labels DETERMINISTIK (loader shuffle=False), aman untuk averaging across seeds.
    """
    model.eval()
    probs_nf, probs_f, labels = [], [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        pixel_values   = batch["pixel_values"].to(DEVICE, non_blocking=True)
        topic          = batch["topic"].to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda"):
            logits, _, _, _ = model(input_ids, attention_mask, pixel_values, topic)
        probs_nf.append(F.softmax(logits.float(), dim=-1)[:, 1].cpu())
        labels.append(batch["label"])

        if tta:
            pv_flip = torch.flip(pixel_values, dims=[3])     # horizontal flip
            with autocast(device_type="cuda"):
                logits_f, _, _, _ = model(input_ids, attention_mask, pv_flip, topic)
            probs_f.append(F.softmax(logits_f.float(), dim=-1)[:, 1].cpu())

    probs_nf = torch.cat(probs_nf).numpy()
    labels   = torch.cat(labels).numpy()
    if tta:
        probs_f = torch.cat(probs_f).numpy()
        return probs_nf, probs_f, labels
    return probs_nf, labels


def train_one_epoch(model, loader, optimizer, scheduler,
                    focal_fn, supcon_fn, scaler, cfg, epoch):
    model.train()
    total_focal = total_supcon = 0.0; n = 0
    g1_log, g2_log = [], []
    supcon_active = epoch > cfg["supcon_warmup_epochs"]

    optimizer.zero_grad()
    pbar = tqdm(loader, desc=f"ep{epoch:02d} {'F+S' if supcon_active else 'F  '}", leave=False)
    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        pixel_values   = batch["pixel_values"].to(DEVICE, non_blocking=True)
        topic          = batch["topic"].to(DEVICE, non_blocking=True)
        labels_b       = batch["label"].to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda"):
            logits, supcon_feats, g1m, g2m = model(input_ids, attention_mask, pixel_values, topic)
            L_focal = focal_fn(logits, labels_b)
            if supcon_active:
                L_supcon = supcon_fn(supcon_feats, labels_b)
                loss_total = L_focal + cfg["supcon_lambda"] * L_supcon
            else:
                L_supcon   = torch.tensor(0.0, device=DEVICE)
                loss_total = L_focal
            loss = loss_total / cfg["grad_accum"]

        scaler.scale(loss).backward()
        bs = labels_b.size(0)
        total_focal  += L_focal.item() * bs
        total_supcon += L_supcon.item() * bs
        n += bs
        g1_log.append(g1m); g2_log.append(g2m)

        if (step + 1) % cfg["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix(focal=f"{L_focal.item():.3f}", supcon=f"{L_supcon.item():.3f}",
                         g1=f"{g1m:.2f}", g2=f"{g2m:.2f}")

    return total_focal/max(n,1), total_supcon/max(n,1), float(np.mean(g1_log)), float(np.mean(g2_log)), supcon_active

## CELL 19 — Per-Seed Training Loop (RESUME-FRIENDLY)

In [19]:
"""
Loop utama: train model untuk setiap seed, simpan checkpoint + probs (val/test, no-flip & flip).
Resume-friendly: kalau probs sudah ada di disk, seed tsb di-skip.
"""

per_seed_results = {}

for seed in CONFIG["seeds"]:
    seed_ckpt_path = CKPT_DIR  / f"model_seed{seed}.pt"
    val_probs_path = PROBS_DIR / f"val_probs_seed{seed}.npz"
    test_probs_path = PROBS_DIR / f"test_probs_seed{seed}.npz"
    train_probs_path = PROBS_DIR / f"train_probs_seed{seed}.npz"
    summary_path   = OUTPUT_DIR / f"seed{seed}_summary.json"

    if val_probs_path.exists() and test_probs_path.exists() and train_probs_path.exists() and summary_path.exists():
        print(f"\n[skip] Seed {seed}: artefak sudah lengkap di disk.")
        with open(summary_path) as f:
            per_seed_results[seed] = json.load(f)
        continue

    print(f"\n{'='*72}\nTRAINING SEED {seed}\n{'='*72}")
    set_seed(seed)

    # Sampler & loader train per-seed (WeightedRandomSampler ber-RNG seed-affected)
    g = torch.Generator(); g.manual_seed(seed)
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_w).double(),
        num_samples=len(sample_w), replacement=True, generator=g,
    )
    dl_train_seed = DataLoader(ds_train, batch_size=CONFIG["batch_size"], sampler=sampler,
                                num_workers=2, pin_memory=True)

    # Model + optim + scaler
    model = ICAMHateModelSupCon(CONFIG).to(DEVICE)
    num_training_steps = math.ceil(len(dl_train_seed) / CONFIG["grad_accum"]) * CONFIG["num_epochs"]
    optimizer, scheduler = build_optimizer_scheduler(model, CONFIG, num_training_steps)
    scaler = GradScaler("cuda")

    best_val_f1, best_epoch, no_improve = -1.0, 0, 0
    seed_log = []
    t_seed = time.time()

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        focal_loss, supcon_loss, g1m, g2m, supcon_active = train_one_epoch(
            model, dl_train_seed, optimizer, scheduler,
            focal_criterion, supcon_criterion, scaler, CONFIG, epoch,
        )
        val_metrics, _, _ = evaluate(model, dl_val, threshold=0.5, desc=f"seed{seed} ep{epoch:02d} val")
        seed_log.append({
            "epoch": epoch, "focal_loss": focal_loss, "supcon_loss": supcon_loss,
            "g1_mean": g1m, "g2_mean": g2m,
            "val_macro_f1": val_metrics["macro_f1"],
            "val_hate_p": val_metrics["hate_p"], "val_hate_r": val_metrics["hate_r"],
            "val_hate_f1": val_metrics["hate_f1"], "val_auc": val_metrics.get("roc_auc", float("nan")),
        })
        print(f"  ep{epoch:02d} {'F+S' if supcon_active else 'F  '}  "
              f"focal={focal_loss:.4f} supcon={supcon_loss:.4f} | "
              f"val_F1={val_metrics['macro_f1']:.4f} hate_F1={val_metrics['hate_f1']:.3f}")

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_epoch, no_improve = val_metrics["macro_f1"], epoch, 0
            torch.save({"epoch": epoch, "state_dict": model.state_dict()}, seed_ckpt_path)
        else:
            no_improve += 1
            if no_improve >= CONFIG["early_stop_pat"]:
                print(f"  early stopping seed {seed} at epoch {epoch}")
                break

    pd.DataFrame(seed_log).to_csv(OUTPUT_DIR / f"seed{seed}_metrics.csv", index=False)

    # Reload best checkpoint
    ckpt = torch.load(seed_ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])

    # Eval val (TTA)
    val_nf, val_f, y_val = evaluate_get_probs(model, dl_val, tta=True, desc=f"seed{seed} val probs")
    np.savez(val_probs_path, probs_no_flip=val_nf, probs_flip=val_f, labels=y_val)

    # Eval test (TTA)
    test_nf, test_f, y_test = evaluate_get_probs(model, dl_test, tta=True, desc=f"seed{seed} test probs")
    np.savez(test_probs_path, probs_no_flip=test_nf, probs_flip=test_f, labels=y_test)

    # Eval train (no TTA, untuk gap diagnosis)
    train_nf, y_train = evaluate_get_probs(model, dl_train_eval, tta=False, desc=f"seed{seed} train probs")
    np.savez(train_probs_path, probs_no_flip=train_nf, labels=y_train)

    # Quick per-seed metrics @ threshold 0.5 (untuk variance analysis)
    per_seed_test_metrics = compute_metrics(y_test, (test_nf >= 0.5).astype(int), test_nf)
    per_seed_results[seed] = {
        "best_epoch":           best_epoch,
        "best_val_macro_f1":    float(best_val_f1),
        "test_macro_f1_thr05":  float(per_seed_test_metrics["macro_f1"]),
        "test_hate_f1_thr05":   float(per_seed_test_metrics["hate_f1"]),
        "test_hate_r_thr05":    float(per_seed_test_metrics["hate_r"]),
        "test_auc":             float(per_seed_test_metrics.get("roc_auc", float("nan"))),
        "elapsed_sec":          float(time.time() - t_seed),
    }
    with open(summary_path, "w") as f:
        json.dump(per_seed_results[seed], f, indent=2)

    print(f"  done seed {seed}: best_val_F1={best_val_f1:.4f} (ep{best_epoch})  "
          f"test_F1@0.5={per_seed_results[seed]['test_macro_f1_thr05']:.4f}  "
          f"({(time.time()-t_seed)/60:.1f} min)")

    # Free memory utk seed berikutnya
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

print(f"\n{'='*72}\nALL SEEDS DONE\n{'='*72}")


TRAINING SEED 42


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

... (259 outputs left out)

seed999 test probs:   0%|          | 0/32 [00:00<?, ?it/s]

seed999 train probs:   0%|          | 0/107 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c713d5a58a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c713d5a58a0>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7c713d5a58a0>
    Traceback (most recent call last):
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

  File "/usr/local/lib/python3.12/dist-packages/t

  done seed 999: best_val_F1=0.7908 (ep4)  test_F1@0.5=0.7860  (11.5 min)

ALL SEEDS DONE


## Load All Seed Probs

In [20]:
all_val_nf, all_val_f   = {}, {}
all_test_nf, all_test_f = {}, {}
all_train_nf            = {}
y_val_ref = y_test_ref = y_train_ref = None

for seed in CONFIG["seeds"]:
    val  = np.load(PROBS_DIR / f"val_probs_seed{seed}.npz")
    test = np.load(PROBS_DIR / f"test_probs_seed{seed}.npz")
    tr   = np.load(PROBS_DIR / f"train_probs_seed{seed}.npz")

    all_val_nf[seed]  = val["probs_no_flip"]; all_val_f[seed]  = val["probs_flip"]
    all_test_nf[seed] = test["probs_no_flip"]; all_test_f[seed] = test["probs_flip"]
    all_train_nf[seed] = tr["probs_no_flip"]

    if y_val_ref is None:
        y_val_ref, y_test_ref, y_train_ref = val["labels"], test["labels"], tr["labels"]
    else:
        assert np.array_equal(y_val_ref,   val["labels"])
        assert np.array_equal(y_test_ref,  test["labels"])
        assert np.array_equal(y_train_ref, tr["labels"])

print(f"Loaded probs from {len(CONFIG['seeds'])} seeds")
print(f"Val: {len(y_val_ref)}  Test: {len(y_test_ref)}  Train: {len(y_train_ref)}")

Loaded probs from 5 seeds
Val: 603  Test: 1005  Train: 3415


## CELL 21 — 1B.9: Ensemble (NO TTA)

In [21]:
# Average probabilitas no-flip across seeds
val_ens_no_tta   = np.mean(np.stack(list(all_val_nf.values()),   axis=0), axis=0)
test_ens_no_tta  = np.mean(np.stack(list(all_test_nf.values()),  axis=0), axis=0)
train_ens_no_tta = np.mean(np.stack(list(all_train_nf.values()), axis=0), axis=0)

# Kalibrasi threshold di val
best_f1_ens_no, best_thr_ens_no = search_threshold(
    y_val_ref, val_ens_no_tta,
    CONFIG["thr_search_lo"], CONFIG["thr_search_hi"], CONFIG["thr_search_step"],
)

# Eval test
y_pred_test_no = (test_ens_no_tta >= best_thr_ens_no).astype(int)
metrics_test_no_tta = compute_metrics(y_test_ref, y_pred_test_no, test_ens_no_tta)
metrics_test_no_tta["threshold"] = best_thr_ens_no

# Train metrics utk gap (pakai threshold sama)
y_pred_train_no = (train_ens_no_tta >= best_thr_ens_no).astype(int)
metrics_train_no_tta = compute_metrics(y_train_ref, y_pred_train_no, train_ens_no_tta)
gap_no_tta = metrics_train_no_tta["macro_f1"] - metrics_test_no_tta["macro_f1"]

print(f"\n{'='*72}\n1B.9 ENSEMBLE 5 SEEDS (NO TTA)\n{'='*72}")
print(f"  Threshold (val-cal) : {best_thr_ens_no:.2f}  (val_macro_f1={best_f1_ens_no:.4f})")
print(f"  Test Macro-F1       : {metrics_test_no_tta['macro_f1']:.4f}")
print(f"  Test Hate P         : {metrics_test_no_tta['hate_p']:.4f}")
print(f"  Test Hate R         : {metrics_test_no_tta['hate_r']:.4f}")
print(f"  Test Hate F1        : {metrics_test_no_tta['hate_f1']:.4f}")
print(f"  Test Weighted-F1    : {metrics_test_no_tta['weighted_f1']:.4f}")
print(f"  Test Accuracy       : {metrics_test_no_tta['accuracy']:.4f}")
print(f"  Test ROC-AUC        : {metrics_test_no_tta.get('roc_auc', float('nan')):.4f}")
print(f"  Train Macro-F1      : {metrics_train_no_tta['macro_f1']:.4f}")
print(f"  Gap (train-test)    : {gap_no_tta:.4f}")
print("\nClassification report:")
print(classification_report(y_test_ref, y_pred_test_no, target_names=["not_hate","hate"], digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test_ref, y_pred_test_no))

threshold search:   0%|          | 0/26 [00:00<?, ?it/s]


1B.9 ENSEMBLE 5 SEEDS (NO TTA)
  Threshold (val-cal) : 0.57  (val_macro_f1=0.8069)
  Test Macro-F1       : 0.8076
  Test Hate P         : 0.7864
  Test Hate R         : 0.7003
  Test Hate F1        : 0.7409
  Test Weighted-F1    : 0.8283
  Test Accuracy       : 0.8308
  Test ROC-AUC        : 0.8875
  Train Macro-F1      : 0.9758
  Gap (train-test)    : 0.1681

Classification report:
              precision    recall  f1-score   support

    not_hate     0.8506    0.8997    0.8744       658
        hate     0.7864    0.7003    0.7409       347

    accuracy                         0.8308      1005
   macro avg     0.8185    0.8000    0.8076      1005
weighted avg     0.8284    0.8308    0.8283      1005

Confusion matrix:
[[592  66]
 [104 243]]


## CELL 22 — 1B.10: Ensemble + TTA Horizontal Flip

In [22]:
# Per seed: average no-flip + flip; kemudian average across seeds
val_ens_tta  = np.mean(np.stack([(all_val_nf[s]  + all_val_f[s])  / 2 for s in CONFIG["seeds"]], axis=0), axis=0)
test_ens_tta = np.mean(np.stack([(all_test_nf[s] + all_test_f[s]) / 2 for s in CONFIG["seeds"]], axis=0), axis=0)

best_f1_ens_tta, best_thr_ens_tta = search_threshold(
    y_val_ref, val_ens_tta,
    CONFIG["thr_search_lo"], CONFIG["thr_search_hi"], CONFIG["thr_search_step"],
)

y_pred_test_tta = (test_ens_tta >= best_thr_ens_tta).astype(int)
metrics_test_tta = compute_metrics(y_test_ref, y_pred_test_tta, test_ens_tta)
metrics_test_tta["threshold"] = best_thr_ens_tta

# Train gap (TTA tidak diaplikasikan di train karena augmentasi bukan tujuan diagnostik)
gap_tta = metrics_train_no_tta["macro_f1"] - metrics_test_tta["macro_f1"]

print(f"\n{'='*72}\n1B.10 ENSEMBLE 5 SEEDS + TTA HORIZONTAL FLIP\n{'='*72}")
print(f"  Threshold (val-cal) : {best_thr_ens_tta:.2f}  (val_macro_f1={best_f1_ens_tta:.4f})")
print(f"  Test Macro-F1       : {metrics_test_tta['macro_f1']:.4f}")
print(f"  Test Hate P         : {metrics_test_tta['hate_p']:.4f}")
print(f"  Test Hate R         : {metrics_test_tta['hate_r']:.4f}")
print(f"  Test Hate F1        : {metrics_test_tta['hate_f1']:.4f}")
print(f"  Test Weighted-F1    : {metrics_test_tta['weighted_f1']:.4f}")
print(f"  Test Accuracy       : {metrics_test_tta['accuracy']:.4f}")
print(f"  Test ROC-AUC        : {metrics_test_tta.get('roc_auc', float('nan')):.4f}")
print(f"  Gap (train-test)    : {gap_tta:.4f}")
print("\nClassification report:")
print(classification_report(y_test_ref, y_pred_test_tta, target_names=["not_hate","hate"], digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test_ref, y_pred_test_tta))

threshold search:   0%|          | 0/26 [00:00<?, ?it/s]


1B.10 ENSEMBLE 5 SEEDS + TTA HORIZONTAL FLIP
  Threshold (val-cal) : 0.63  (val_macro_f1=0.8084)
  Test Macro-F1       : 0.8025
  Test Hate P         : 0.8195
  Test Hate R         : 0.6542
  Test Hate F1        : 0.7276
  Test Weighted-F1    : 0.8256
  Test Accuracy       : 0.8308
  Test ROC-AUC        : 0.8875
  Gap (train-test)    : 0.1733

Classification report:
              precision    recall  f1-score   support

    not_hate     0.8352    0.9240    0.8773       658
        hate     0.8195    0.6542    0.7276       347

    accuracy                         0.8308      1005
   macro avg     0.8273    0.7891    0.8025      1005
weighted avg     0.8298    0.8308    0.8256      1005

Confusion matrix:
[[608  50]
 [120 227]]


## CELL 23 — Per-Seed Variance Analysis

In [23]:
print(f"\n{'='*72}\nPER-SEED ANALYSIS (untuk variance & sanity check)\n{'='*72}")
df_per_seed = pd.DataFrame(per_seed_results).T
df_per_seed.index.name = "seed"
print(df_per_seed.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# Per-seed test Macro-F1 mean and std
seed_f1s = [r["test_macro_f1_thr05"] for r in per_seed_results.values()]
print(f"\nTest Macro-F1 (per-seed @ threshold 0.5):")
print(f"  Mean: {np.mean(seed_f1s):.4f}")
print(f"  Std : {np.std(seed_f1s):.4f}")
print(f"  Min : {np.min(seed_f1s):.4f}")
print(f"  Max : {np.max(seed_f1s):.4f}")
print(f"\nEnsemble gain vs mean single seed:")
print(f"  No TTA  : +{metrics_test_no_tta['macro_f1'] - np.mean(seed_f1s):.4f}")
print(f"  With TTA: +{metrics_test_tta['macro_f1']    - np.mean(seed_f1s):.4f}")


PER-SEED ANALYSIS (untuk variance & sanity check)
      best_epoch  best_val_macro_f1  test_macro_f1_thr05  test_hate_f1_thr05  test_hate_r_thr05  test_auc  elapsed_sec
seed                                                                                                                  
42       13.0000             0.7893               0.7976              0.7418             0.7781    0.8854    1028.7882
123       9.0000             0.7979               0.7958              0.7367             0.7579    0.8855     980.3709
2024     12.0000             0.7988               0.7915              0.7201             0.6859    0.8792     973.6497
7        15.0000             0.8040               0.7967              0.7358             0.7464    0.8855    1014.0898
999       4.0000             0.7908               0.7860              0.7160             0.6974    0.8755     691.6054

Test Macro-F1 (per-seed @ threshold 0.5):
  Mean: 0.7935
  Std : 0.0043
  Min : 0.7860
  Max : 0.7976

Ensemble gai

## CELL 24 — Save Final Artifacts + Summary

In [24]:
final_results = {
    "exp_id": CONFIG["exp_id"],
    "seeds":  CONFIG["seeds"],

    "1B9_ensemble_no_tta": {
        "threshold":      float(best_thr_ens_no),
        "val_macro_f1":   float(best_f1_ens_no),
        "test_metrics":   {k: float(v) for k, v in metrics_test_no_tta.items()},
        "train_metrics":  {k: float(v) for k, v in metrics_train_no_tta.items()},
        "gap_train_test": float(gap_no_tta),
    },
    "1B10_ensemble_with_tta": {
        "threshold":      float(best_thr_ens_tta),
        "val_macro_f1":   float(best_f1_ens_tta),
        "test_metrics":   {k: float(v) for k, v in metrics_test_tta.items()},
        "gap_train_test": float(gap_tta),
    },
    "per_seed_results": per_seed_results,
    "variance_analysis": {
        "test_macro_f1_per_seed_mean": float(np.mean(seed_f1s)),
        "test_macro_f1_per_seed_std":  float(np.std(seed_f1s)),
        "ensemble_gain_no_tta":   float(metrics_test_no_tta["macro_f1"] - np.mean(seed_f1s)),
        "ensemble_gain_with_tta": float(metrics_test_tta["macro_f1"]    - np.mean(seed_f1s)),
    },
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
with open(OUTPUT_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print(f"\n{'='*72}\nFINAL SUMMARY - KIRIMKAN BLOK INI KE CHAT\n{'='*72}")
print(f"Experiment        : {CONFIG['exp_id']}")
print(f"Seeds             : {CONFIG['seeds']}")
print()
print(f"--- 1B.9 ENSEMBLE NO TTA ---")
print(f"Threshold         : {best_thr_ens_no:.2f}")
print(f"Macro-F1          : {metrics_test_no_tta['macro_f1']:.4f}")
print(f"Hate P/R/F1       : {metrics_test_no_tta['hate_p']:.4f} / {metrics_test_no_tta['hate_r']:.4f} / {metrics_test_no_tta['hate_f1']:.4f}")
print(f"Weighted-F1 / Acc : {metrics_test_no_tta['weighted_f1']:.4f} / {metrics_test_no_tta['accuracy']:.4f}")
print(f"ROC-AUC           : {metrics_test_no_tta.get('roc_auc', float('nan')):.4f}")
print(f"Gap train-test    : {gap_no_tta:.4f}")
print()
print(f"--- 1B.10 ENSEMBLE + TTA ---")
print(f"Threshold         : {best_thr_ens_tta:.2f}")
print(f"Macro-F1          : {metrics_test_tta['macro_f1']:.4f}")
print(f"Hate P/R/F1       : {metrics_test_tta['hate_p']:.4f} / {metrics_test_tta['hate_r']:.4f} / {metrics_test_tta['hate_f1']:.4f}")
print(f"Weighted-F1 / Acc : {metrics_test_tta['weighted_f1']:.4f} / {metrics_test_tta['accuracy']:.4f}")
print(f"ROC-AUC           : {metrics_test_tta.get('roc_auc', float('nan')):.4f}")
print(f"Gap train-test    : {gap_tta:.4f}")
print()
print(f"--- PER-SEED VARIANCE ---")
print(f"Mean / Std        : {np.mean(seed_f1s):.4f} / {np.std(seed_f1s):.4f}")
print(f"Range             : [{np.min(seed_f1s):.4f}, {np.max(seed_f1s):.4f}]")
print(f"Ensemble gain     : +{metrics_test_no_tta['macro_f1'] - np.mean(seed_f1s):.4f} (no TTA)  +{metrics_test_tta['macro_f1'] - np.mean(seed_f1s):.4f} (with TTA)")
print(f"\nArtifacts saved to: {OUTPUT_DIR}")


FINAL SUMMARY - KIRIMKAN BLOK INI KE CHAT
Experiment        : 1B.9_ensemble_5seeds
Seeds             : [42, 123, 2024, 7, 999]

--- 1B.9 ENSEMBLE NO TTA ---
Threshold         : 0.57
Macro-F1          : 0.8076
Hate P/R/F1       : 0.7864 / 0.7003 / 0.7409
Weighted-F1 / Acc : 0.8283 / 0.8308
ROC-AUC           : 0.8875
Gap train-test    : 0.1681

--- 1B.10 ENSEMBLE + TTA ---
Threshold         : 0.63
Macro-F1          : 0.8025
Hate P/R/F1       : 0.8195 / 0.6542 / 0.7276
Weighted-F1 / Acc : 0.8256 / 0.8308
ROC-AUC           : 0.8875
Gap train-test    : 0.1733

--- PER-SEED VARIANCE ---
Mean / Std        : 0.7935 / 0.0043
Range             : [0.7860, 0.7976]
Ensemble gain     : +0.0141 (no TTA)  +0.0089 (with TTA)

Artifacts saved to: /kaggle/working/exp_1B9_ensemble


In [25]:
import shutil
import os
from IPython.display import FileLink

# 1. Tentukan path folder yang ingin di-download
folder_path = '/kaggle/working/exp_1B9_ensemble'
# 2. Tentukan nama file zip outputnya
output_zip = 'exp_1B9_ensemble_results'

# 3. Proses zipping
if os.path.exists(folder_path):
    shutil.make_archive(output_zip, 'zip', folder_path)
    print(f"✅ Berhasil men-zip folder: {output_zip}.zip")
    
    # 4. Munculkan link download langsung
    display(FileLink(f"{output_zip}.zip"))
else:
    print(f"❌ Error: Folder {folder_path} tidak ditemukan.")

✅ Berhasil men-zip folder: exp_1B9_ensemble_results.zip


/kaggle/working/exp_1B9_ensemble_results.zip